# CMU 11-768 Assignment 1 · Agent Harness Walkthrough

A Colab companion to Russell's independent adaptation of Assignment 1. Follow one request through the harness, see how the agent saves room in its context, then watch the same loop operate on a chess board.

[Assignment 1 repository](https://github.com/xqzr2000/cmu-11-768-assignment-1) · [Original CMU specification](https://github.com/cmu-agents/assignment-1/blob/main/ASSIGNMENT.md)

**Run in Colab.** Use the default Python 3 CPU runtime. Run the setup cell, then step through with **Shift + Enter**. Setup fetches the repository and installs Python packages. The exercises use the checked-out implementation with scripted model replies, a fake terminal, and an in-memory chess API. They require zero model API calls and zero Docker or Modal runs.

| Chapter | What you will run |
| --- | --- |
| 1 · Coding agent | Inspect the prompt and tool schemas, recover from invalid calls, load a skill, submit a toy patch, and read the saved trajectory |
| 2 · Context compaction | Compare the same scripted run with compaction on and off, then calculate what changed in the recorded Django runs |
| 3 · Chess agent | Compare two views of a board, follow UCI moves through the API, simulate a move, and inspect the handoff to sandboxed Python |
| 4 · Project results | Connect the recorded tests and experiments to the original requirements |

## 0 · Set up the source checkout

The checkout is pinned to the Assignment 1 code reviewed for this walkthrough. The two local packages are installed together, as they are in the repo's `uv` workspace. The `python-chess` dependency implements chess rules for the notebook's in-memory examples. The actual project's Docker workflow is described in the repository README.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import subprocess
import sys
import tomllib

REPO_URL = "https://github.com/xqzr2000/cmu-11-768-assignment-1.git"
SOURCE_COMMIT = "ca87bbef76059287468d82fb37ea01aa9646c31d"
workspace = TemporaryDirectory(prefix="cmu-agent-walkthrough-")
repo = Path(workspace.name) / "assignment-1"

subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPO_URL, str(repo)], check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_COMMIT], cwd=repo, check=True)
runtime_package = repo / "packages" / "agent-sandbox-runtime"
lock = tomllib.loads((repo / "uv.lock").read_text(encoding="utf-8"))
example_dependencies = {"httpx", "jinja2", "openai", "python-dotenv", "pyyaml", "tqdm"}
pins = [
    f"{package['name']}=={package['version']}"
    for package in lock["package"]
    if package["name"] in example_dependencies
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     str(repo), str(runtime_package), *pins, "python-chess==1.999"],
    check=True,
)
print(f"Assignment 1 source loaded at {SOURCE_COMMIT[:7]}")

### How the repository is wired

`make` targets call functions in `src/assignment/cli.py`. The CLI loads a task, starts an environment, constructs an agent, and writes artifacts. `Agent.run` owns the common action and observation loop. `CodeAgent` and `ChessAgent` provide their own prompts, tools, and definition of `finished`.

In the CMU starter, `Environment` uses Modal sandboxes. In this fork, `src/assignment/env.py` delegates to `Sandbox` in `packages/agent-sandbox-runtime/`. Codespaces hosts the Docker workflow, and model inference goes to an OpenAI-compatible provider. Colab uses two small stand-ins at those boundaries, so we can inspect the harness one step at a time.

| CMU requirement | Implementation in this fork |
| --- | --- |
| 1.1–1.2 · Build prompts and run ReAct | `agent/base.py` builds request history and drives the loop. `agent/code_agent.py` supplies coding prompts and environment details. |
| 1.3–1.5 · Run coding tools, disclose skills, verify a patch | `agent/code_agent.py` dispatches tools, `agent/base.py` loads skills, and `scripts/evaluate.py` replays the submitted patch in a fresh testbed. |
| 2.1–2.2 · Compact old context and measure the effect | `agent/base.py` summarizes older steps before the next action request. `scripts/report_token_usage.py` compares saved usage. |
| 3.1–3.3 · Play, vary observations, and simulate | `agent/tools.py` declares schemas. `agent/chess_agent.py` formats observations and tracks state. `agent/chess_tools.py` calls the chess API. |
| 3.4–3.5 · Programmatic tools and a chess skill | `agent/chess_tools.py` hands off to `sandbox_python.py` inside Docker. `tasks/chess-skills/select-move/SKILL.md` describes the search strategy. |

Paths beginning with `agent/` in this table are under `src/assignment/`. Browse the [source checkout](https://github.com/xqzr2000/cmu-11-768-assignment-1/tree/ca87bbef76059287468d82fb37ea01aa9646c31d).

In [ ]:
import base64
from copy import deepcopy
import inspect
import json
import os
import runpy
from types import SimpleNamespace
from unittest.mock import patch

import chess
import httpx
from IPython.display import Markdown, display
from openai.types.chat import ChatCompletion

from assignment.agent import Agent, ChessAgent, CodeAgent, StepLimitError, format_chess_state
from assignment.cli import _record_final_observations
from assignment.task import Task

output_dir = Path(workspace.name) / "demo-artifacts"
output_dir.mkdir(exist_ok=True)


def tool_call(call_id, name, arguments):
    return {
        "id": call_id,
        "type": "function",
        "function": {
            "name": name,
            "arguments": arguments if isinstance(arguments, str) else json.dumps(arguments),
        },
    }


def action(*calls):
    return {"role": "assistant", "content": None, "tool_calls": list(calls)}


def scripted_agent(agent_class, *args, actions, summary=None, **kwargs):
    pending = iter(actions)

    def create(**request):
        if "tools" in request:
            message = next(pending)
        else:
            if summary is None:
                raise AssertionError("Unexpected compaction request")
            message = {"role": "assistant", "content": summary}
        return ChatCompletion.model_validate({
            "id": "chatcmpl-colab-demo",
            "object": "chat.completion",
            "created": 0,
            "model": "scripted-colab",
            "choices": [{
                "index": 0,
                "message": message,
                "finish_reason": "tool_calls" if message.get("tool_calls") else "stop",
            }],
        })

    client = SimpleNamespace(chat=SimpleNamespace(completions=SimpleNamespace(create=create)))
    with patch("assignment.agent.base.OpenAI", return_value=client), patch.dict(
        os.environ, {"OPENAI_API_KEY": "unused-colab-placeholder", "OPENAI_MAX_RETRIES": "0"}
    ):
        return agent_class(*args, model="scripted-colab", **kwargs)


class ToyTerminal:
    cwd = "/testbed"
    machine, release, system, version = "x86_64", "demo", "Linux", "Colab fixture"

    def __init__(self, replies=None):
        self.replies = replies or {}
        self.commands = []
        self.stopped = False

    def execute(self, command, **options):
        self.commands.append((command, options))
        if command in self.replies:
            return {"output": self.replies[command], "returncode": 0}
        return {"output": "Command absent from the demo fixtures", "returncode": 1}

    def stop(self):
        self.stopped = True

`scripted_agent` replaces the API client before the repo constructs it. Its replies still use the SDK's `ChatCompletion` type, which lets the real `query_language_model` and `process_response` run. `ToyTerminal.execute` returns fixture text for known commands. These examples exercise the harness without needing a real shell.

## 1 · Trace the coding agent

### 1.1 · Build the first request

`Task.load` reads the task metadata and issue text. `CodeAgent.__init__` builds the system prompt with the sandbox platform and skill catalog and places the issue and working directory in the user message. The tool schemas are passed separately through the API's `tools` field.

Run the next cell to inspect those pieces. The actual assignment task appears as a short excerpt. The scripted run works on a toy change to a greeting.

In [ ]:
task = Task.load(repo / "tasks" / "chess-terminal-move")
skill_path = str(repo / "tasks" / "code-skills")
toy_task = "Update the greeting in hello.py to say Hello, Colab."

toy_patch = (
    "diff --git a/hello.py b/hello.py\n"
    "--- a/hello.py\n+++ b/hello.py\n"
    "@@ -1 +1 @@\n-print('Hello')\n+print('Hello, Colab')\n"
)
terminal = ToyTerminal({
    "python hello.py": "Hello\n",
    "update-greeting-demo": "Updated hello.py in the fixture.",
    "git diff -- hello.py > patch.txt": "",
    "cat patch.txt": toy_patch,
})

code_path = output_dir / "code-trajectory.json"
code_agent = scripted_agent(
    CodeAgent, toy_task, terminal,
    skills_path=skill_path, logs_save_path=str(code_path), step_limit=7,
    actions=[
        {"role": "assistant", "content": "I will inspect the greeting first."},
        action(
            tool_call("bad-json", "execute", "{"),
            tool_call("unknown", "missing_tool", {}),
            tool_call("inspect", "execute", {"command": "python hello.py"}),
        ),
        action(tool_call("skill", "invoke_skill", {"name": "submit-task"})),
        action(tool_call("edit", "execute", {"command": "update-greeting-demo"})),
        action(tool_call("write-patch", "execute", {"command": "git diff -- hello.py > patch.txt"})),
        action(tool_call("review-patch", "execute", {"command": "cat patch.txt"})),
        action(tool_call("done", "send_message", {"summary": "Verified the greeting patch."})),
    ],
)
initial_prompt = code_agent.build_prompt()
print("Assignment task:", task.id, "·", task.problem_statement.splitlines()[0])
print("Initial message roles:", [message["role"] for message in initial_prompt])
print("Tools advertised:", [tool["function"]["name"] for tool in code_agent.tools])
print(code_agent.system_prompt.split("<system_information>", 1)[1].split("</system_information>", 1)[0].strip())
print(code_agent.skills["submit-task"]["metadata"])
assert code_agent.skills["submit-task"]["content"] not in code_agent.system_prompt

### 1.2–1.4 · Follow actions, tool observations, and completion

1. `Agent.run` calls `maybe_compact_context` before asking the model for an action. Compaction defaults to off here.
2. `query_language_model` saves a prompt snapshot, asks for an action, and increments `steps_taken`. When the response contains just text, the loop appends a recovery message asking for a tool call.
3. `CodeAgent.execute_tool_calls` validates each call. Bad JSON and unknown tools yield tagged error observations. A valid `execute` goes to `env.execute`.
4. `_link_observations` puts tool replies in call order with matching `tool_call_id` values. Those messages become part of the next request.
5. `invoke_skill` returns the full skill. `send_message` sets `finished` and retrieves the submitted patch. The loop saves the trajectory and releases its environment.

After the simulated edit, the script creates the patch, reads it back, and submits it in three separate actions, as the skill specifies.

In [ ]:
code_agent.run()
code_trace = json.loads(code_path.read_text(encoding="utf-8"))
recovery = code_trace["prompts"][1][-1]
print("\nRecovery message:", recovery["content"])
print("\nObservations in the next request:")
for message in code_trace["prompts"][2][-3:]:
    print(f"{message['tool_call_id']:>9}  {message['content'].replace(chr(10), ' ')}")

skill_reply = next(message for message in code_agent.history if message.get("tool_call_id") == "skill")
print("\nSkill loaded:", skill_reply["content"].splitlines()[1])
print("Illustrative patch:\n" + code_agent.submitted_patch)
print("Finished:", code_agent.finished, "| cleanup:", terminal.stopped, "| steps:", len(code_trace["responses"]))

assert [message["tool_call_id"] for message in code_trace["prompts"][2][-3:]] == [
    "bad-json", "unknown", "inspect",
]
assert terminal.commands[0][0] == "python hello.py"
assert code_agent.submitted_patch == toy_patch and terminal.stopped

### 1.5 · Verify the real deliverable

In Codespaces, `run_code_agent` exports `patch.txt` as `artifacts/fix.patch` after the run. `make check-part1` applies the patch in a fresh testbed and grades it independently. A **FAIL_TO_PASS** test catches the original issue and should pass after repair. **PASS_TO_PASS** tests check behavior that was already working.

The recorded Part 1 run took 13 action steps and passed **1/1 FAIL_TO_PASS** and **9/9 PASS_TO_PASS** tests. On a game-ending White move, the repaired app skips Black's reply because the game already has an outcome. The patch printed above is a fixture for studying the tool protocol. The actual chess run is documented in the [learning log](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/docs/Learning-Progress.md#3-what-weve-finished).

## 2 · Keep useful context as the transcript grows

### 2.1–2.2 · Compare full history and working memory

`maybe_compact_context` checks the estimated prompt size and whether enough assistant steps exist to retire an older prefix. The estimate uses serialized message length, with the latest provider prompt count as a calibration point when available. `compact_context` asks for a short working memory from the older steps, preserves the most recent complete assistant/tool group, and inserts the memory after the original system and task messages. The trajectory still holds the original API snapshots and records the summary request separately.

Run the same three actions twice. A threshold of 1 forces a check as soon as there are enough old steps, and the long first observation makes the difference visible. The short summary is scripted so you can inspect exactly what survives.

In [ ]:
def run_context_example(threshold):
    context_terminal = ToyTerminal({
        "cat notes.txt": "Target file: hello.py.\n" + "Repeated diagnostic output.\n" * 180,
        "python hello.py": "Hello, Colab\n",
        "cat patch.txt": toy_patch,
    })
    agent = scripted_agent(
        CodeAgent, toy_task, context_terminal,
        step_limit=3, compact_threshold_tokens=threshold,
        actions=[
            action(tool_call("old", "execute", {"command": "cat notes.txt"})),
            action(tool_call("recent", "execute", {"command": "python hello.py"})),
            action(tool_call("done", "send_message", {"summary": "Greeting confirmed."})),
        ],
        summary="Target file: hello.py. The notes identify the file to change.",
    )
    agent.run()
    return agent


full = run_context_example(None)
compact = run_context_example(1)
full_prompt, compact_prompt = full.api_prompts[-1], compact.api_prompts[-1]
event, = compact.compaction_events

print("\nBefore the final action:")
for label, prompt in [("Full", full_prompt), ("Compacted", compact_prompt)]:
    print(f"{label:>9} · {len(prompt)} messages ·", [m["role"] for m in prompt])
print("Estimated message tokens before/after:", event["estimated_tokens_before"], event["estimated_tokens_after"])
print(compact_prompt[2]["content"].split("<working_memory>", 1)[1].split("</working_memory>", 1)[0].strip())

assert compact_prompt[:2] == full_prompt[:2]
assert compact_prompt[-2:] == full_prompt[-2:]
assert compact_prompt[-1]["tool_call_id"] == "recent"
assert all(message.get("tool_call_id") != "old" for message in compact_prompt)
assert any("Repeated diagnostic output" in m["content"] for m in compact.api_prompts[1])
assert event["estimated_tokens_after"] < event["estimated_tokens_before"]

The two prompts share the original instructions and the complete latest action and observation. In the compacted prompt, a short memory takes the place of the old action and long tool output. The earlier API snapshot still contains that tool output, so it can be audited later.

### Recorded SWE-bench run · Fewer tokens and fewer cache hits

The [token report](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/docs/reports/token-usage-analysis.md) compares two runs on `django__django-15368` with the same model. Both resolved the task and passed **1/1 FAIL_TO_PASS** and **29/29 PASS_TO_PASS**. The compacted run used a threshold of 6,000 and took 20 action steps plus two summary requests. The baseline took 19 action steps.

The peak prompt below is the largest action prompt. Total tokens also include summary requests. Uncached input is total input minus cached input.

In [ ]:
usage = {
    "Compacted": {"peak": 5976, "input": 98644, "cached": 80896, "output": 7159},
    "Full context": {"peak": 11400, "input": 135766, "cached": 128896, "output": 4717},
}
for values in usage.values():
    values["total"] = values["input"] + values["output"]
    values["uncached"] = values["input"] - values["cached"]

compact_usage, full_usage = usage["Compacted"], usage["Full context"]
display(Markdown(
    "| Metric | Compacted | Full context |\n| --- | ---: | ---: |\n"
    + "\n".join(
        f"| {label} | {compact_usage[key]:,} | {full_usage[key]:,} |"
        for label, key in [("Peak action prompt", "peak"), ("Total tokens", "total"),
                           ("Cached input tokens", "cached"), ("Uncached input tokens", "uncached"),
                           ("Output tokens", "output")]
    )
))
total_reduction = 1 - compact_usage["total"] / full_usage["total"]
uncached_ratio = compact_usage["uncached"] / full_usage["uncached"]
cache_break_even = (compact_usage["uncached"] - full_usage["uncached"]) / (
    full_usage["cached"] - compact_usage["cached"]
)
print(f"Total tokens fell {total_reduction:.1%}. Uncached input grew {uncached_ratio:.1f}×.")
print(f"Input-only break-even cached rate: {cache_break_even:.1%} of the uncached rate.")

Writing a new working memory changes the shared prompt prefix, which helps explain the lost cache hits. At the input-only break-even rate, the extra uncached input costs as much as the saved cached input. Completion tokens and provider rates also matter when comparing dollar cost. These are two recorded runs on one task.

## 3 · Reuse the agent loop for chess

### 3.1–3.2 · Observe the board and play a move

A **UCI move** such as `e2e4` gives the start and destination squares. A **FEN** gives the position, side to move, castling and en passant information, and move counters. The real chess agent gets its initial state from `POST /api/reset`. Its `play_move` tool sends UCI to `POST /api/move`, updates `last_state`, and sets `finished` from `game_over`. In the production app, Black replies automatically if the game continues.

The notebook's in-memory API handles the same endpoints using `python-chess`. It gives Black a deterministic reply for reproducibility. The first cell builds a toy server and sends both observation formats the same underlying board. Look for the list of legal actions at the bottom of the second view.

In [ ]:
class ChessFixture:
    def __init__(self, initial_fen=chess.STARTING_FEN):
        self.initial_fen = initial_fen
        self.board = chess.Board(initial_fen)
        self.requests = []

    @staticmethod
    def state(board, human_move=None, engine_move=None):
        outcome = board.outcome(claim_draw=True)
        return {
            "fen": board.fen(),
            "squares": {chess.square_name(square): piece.symbol()
                        for square, piece in board.piece_map().items()},
            "turn": chess.COLOR_NAMES[board.turn],
            "status": "Game over" if outcome else "In progress",
            "in_check": board.is_check(),
            "game_over": outcome is not None,
            "legal_moves": sorted(move.uci() for move in board.legal_moves),
            "human_move": human_move,
            "engine_move": engine_move,
        }

    def handle(self, request):
        route = (request.method, request.url.path)
        self.requests.append(route)
        payload = json.loads(request.content) if request.content else {}
        try:
            if route == ("POST", "/api/reset"):
                self.board = chess.Board(self.initial_fen)
            elif route == ("POST", "/api/simulate"):
                candidate = chess.Board(payload["fen"])
                if payload.get("move"):
                    candidate.push_uci(payload["move"])
                return httpx.Response(200, json=self.state(candidate))
            elif route == ("POST", "/api/move"):
                human_move = payload["move"]
                self.board.push_uci(human_move)
                engine_move = None
                if not self.board.is_game_over(claim_draw=True):
                    engine_move = min(move.uci() for move in self.board.legal_moves)
                    self.board.push_uci(engine_move)
                return httpx.Response(200, json=self.state(self.board, human_move, engine_move))
            elif route != ("GET", "/api/state"):
                return httpx.Response(404, json={"detail": "Unknown fixture route"})
        except (ValueError, KeyError) as exc:
            return httpx.Response(400, json={"detail": str(exc)})
        return httpx.Response(200, json=self.state(self.board))

    def client(self):
        return httpx.Client(base_url="http://chess-demo.test", transport=httpx.MockTransport(self.handle))


opening = ChessFixture.state(chess.Board())
board_only = format_chess_state(opening, include_legal_moves=False)
board_and_moves = format_chess_state(opening, include_legal_moves=True)
print("BOARD ONLY")
print(board_only)
print("\nADDED WHEN LEGAL MOVES ARE ENABLED")
print(next(line for line in board_and_moves.splitlines() if line.startswith("legal_moves:")))
assert "legal_moves:" not in board_only
assert len(opening["legal_moves"]) == 20

Try an illegal opening move followed by a legal one. A second live move in the same assistant response is rejected before making an HTTP request, since the first live move and Black's reply can change the position. The third scripted action ends the example with a step-budget stop. A separate fixture starts one move before checkmate to show the terminal condition directly.

The trajectory evaluator matches the call ID to its tool message. The CLI appends `final_observations` because there is no following prompt to capture the last replies. The notebook applies that same helper and uses the real offline evaluator.

In [ ]:
def run_chess_example(initial_fen, actions, expected_terminal, label):
    fixture = ChessFixture(initial_fen)
    trajectory_path = output_dir / f"{label}-trajectory.json"
    result_path = output_dir / f"{label}-result.json"
    with fixture.client() as client:
        agent = scripted_agent(
            ChessAgent, ToyTerminal(), http_client=client,
            actions=actions, step_limit=len(actions), include_legal_moves=True,
            logs_save_path=str(trajectory_path),
        )
        try:
            agent.run()
        except StepLimitError:
            print("Reached the scripted step budget.")
        _record_final_observations(trajectory_path, agent)

    assert agent.finished is expected_terminal
    result_path.write_text(json.dumps(agent.last_state), encoding="utf-8")
    evaluate = runpy.run_path(str(repo / "scripts" / "eval_chess_trajectory.py"))["evaluate"]
    return agent, fixture.requests, evaluate(trajectory_path, result_path)


opening_agent, opening_requests, opening_score = run_chess_example(
    chess.STARTING_FEN,
    [
        action(tool_call("illegal", "play_move", {"move": "e2e5"})),
        action(tool_call("white-1", "play_move", {"move": "e2e4"}),
               tool_call("parallel-1", "play_move", {"move": "g1f3"})),
        action(tool_call("white-2", "play_move", {"move": "g1f3"})),
    ],
    expected_terminal=False,
    label="opening",
)
mate_agent, mate_requests, mate_score = run_chess_example(
    "7k/5Q2/6K1/8/8/8/8/8 w - - 0 1",
    [action(tool_call("mate", "play_move", {"move": "f7g7"}))],
    expected_terminal=True,
    label="checkmate",
)
print("\nOpening HTTP requests:", opening_requests)
print("Opening move outcomes:", [(entry["move"], entry["outcome"]) for entry in opening_score["moves"]])
print("Checkmate · game_over:", mate_score["game_over"], "| Black's reply:", mate_agent.last_state["engine_move"])
assert (opening_score["accepted_moves"], opening_score["tool_errors"], opening_score["unknown_outcomes"]) == (2, 2, 0)
assert opening_requests.count(("POST", "/api/move")) == 3
assert mate_score["accepted_moves"] == 1 and mate_agent.last_state["engine_move"] is None

The opening example has four `play_move` calls, three server requests, and two accepted moves. One move fails chess validation and the extra parallel call fails the harness's ordering rule. In the checkmate example, the same loop stops immediately because the observation contains `game_over: true`.

### 3.3 · Simulate without changing the live board

`simulate_move(fen, move)` sends the supplied FEN to `POST /api/simulate` and returns a position after **one ply**, a move by one side. You can chain those results to explore both sides. A live `play_move` can advance two plies because it includes Black's reply.

In [ ]:
sim_fixture = ChessFixture()
with sim_fixture.client() as client:
    sim_agent = scripted_agent(
        ChessAgent, ToyTerminal(), http_client=client, actions=[], programmatic_tools=True,
        auto_stop_environment=False,
    )
    simulated_reply, = sim_agent.execute_tool_calls([
        tool_call("what-if", "simulate_move", {"fen": opening["fen"], "move": "e2e4"}),
    ])
    simulated = json.loads(simulated_reply["content"])
    print("Simulated position:", simulated["fen"])
    print("After simulate · live side to move:", sim_agent.last_state["turn"])
    sim_agent.execute_tool_calls([tool_call("commit", "play_move", {"move": "e2e4"})])
    print("After play     · Black's response:", sim_agent.last_state["engine_move"])
    print("After play     · live side to move:", sim_agent.last_state["turn"])

assert simulated["turn"] == "black"
assert sim_fixture.requests == [("POST", "/api/reset"), ("POST", "/api/simulate"), ("POST", "/api/move")]
assert sim_agent.last_state["fen"] != simulated["fen"]

### 3.4–3.5 · Load a search skill and see the sandbox handoff

The `select-move` skill gives the agent a short list of opening moves, then teaches it to search candidate lines with `simulate_move` and commit once with `play_move`. The production `run_python` tool passes the snippet to the sandbox runner as base64, which keeps shell quoting predictable. The sandbox runner captures `stdout`, `stderr`, and an `error` field. After it returns, `ChessAgent` fetches the live board so the next model request sees the board after any committed move.

In this example, the stand-in records what would be sent to the sandbox and returns a scripted runner response. It also advances the fixture board to represent the one committed move. The code reproduces the handoff and refresh using the actual `ChessAgent` dispatcher.

In [ ]:
class ToyPythonSandbox:
    def __init__(self, fixture):
        self.fixture = fixture
        self.calls = []

    def execute(self, command, **options):
        self.calls.append((command, options))
        # Represent the one move the canned snippet commits after its search.
        self.fixture.board.push_uci("e2e4")
        self.fixture.board.push_uci("e7e5")
        result = {"stdout": "Selected e2e4\n", "stderr": "", "error": None}
        return {"returncode": 0, "stdout": json.dumps(result)}


skill_fixture = ChessFixture()
sandbox = ToyPythonSandbox(skill_fixture)
with skill_fixture.client() as client:
    skill_agent = scripted_agent(
        ChessAgent, sandbox, http_client=client, actions=[], programmatic_tools=True,
        skills_path=str(repo / "tasks" / "chess-skills"), auto_stop_environment=False,
    )
    registered = [tool["function"]["name"] for tool in skill_agent.tools]
    loaded, = skill_agent.execute_tool_calls([tool_call("load-skill", "invoke_skill", {"name": "select-move"})])
    fen = skill_agent.last_state["fen"]
    snippet = (
        f"candidate = simulate_move({fen!r}, 'e2e4')\n"
        "play_move('e2e4')\n"
        "print('Selected e2e4')"
    )
    runner_reply, = skill_agent.execute_tool_calls([tool_call("run-search", "run_python", {"code": snippet})])

command, options = sandbox.calls[0]
command_prefix, encoded = command.rsplit(" ", 1)
print("Registered tools:", registered)
print("Loaded skill:", loaded["content"].splitlines()[1])
print("Sandbox command:", command_prefix, "<base64 snippet>", f"(timeout: {options['timeout']}s)")
print("Decoded payload:\n" + base64.b64decode(encoded).decode())
print("Runner result:", runner_reply["content"].split("\n\n", 1)[0])
print("Last API call:", skill_fixture.requests[-1])

assert len(set(registered)) == len(registered)
assert base64.b64decode(encoded).decode() == snippet
assert skill_fixture.requests[-1] == ("GET", "/api/state")
assert skill_agent.last_state["fen"] == skill_fixture.board.fen()
assert "Live board after the snippet:" in runner_reply["content"]

The decoded payload is a fixed demonstration of the call sequence. The [recorded skill run](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/README.md#10-running-the-chess-agent) used model-generated snippets to search two plies and commit one move per snippet, with the skill's opening shortcut on the first move. The material-only evaluator tied many quiet moves and broke ties by UCI order, which helps explain why the bishop moved repeatedly.

### Recorded chess experiment · What changed when you exposed legal moves?

The [observation report](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/docs/reports/observation-experiment.md) compares one 15-step run per condition for each model. The project used `gpt-5-mini` in the assignment's GPT-OSS-120b slot.

| Model | Board only | Board + legal moves | Output tokens per step, board only → with list |
| --- | ---: | ---: | ---: |
| DeepSeek-V4.1-Flash | 0/15 rejected | 0/15 rejected | 1,277 → 7,817 |
| gpt-5-mini | 3/15 rejected (20%) | 0/15 rejected | 480 → 249 |

For `gpt-5-mini`, the board-only run included one UCI formatting error, one stale king position after castling, and one move that broke a bishop's movement rule. Its run with the list had zero errors. DeepSeek already made zero errors in both runs, and its recorded output tokens per step rose about sixfold with the list. The runs diverged into different positions and none produced a terminal result. These are observations from four short runs.

## 4 · Read the evidence against the original requirements

| Part | Recorded project evidence | What the Colab example exercised |
| --- | --- | --- |
| 1 · Coding agent and repair | Chess patch resolved in 13 steps. Independent replay passed 1/1 FAIL_TO_PASS and 9/9 PASS_TO_PASS. | Prompt construction, text-only recovery, argument validation, tool-call IDs, skill loading, patch submission, and cleanup. |
| 2 · Context compaction | The compacted and full-context Django runs both resolved, with different prompt sizes and cache usage. | A scripted summary replaces an old step while preserving the latest assistant/tool exchange. |
| 3.1 · Full chess game | The real project stopped after two partial games. One reached king and queen against king and bishop before the sandbox expired. | A fixture plays opening moves and a separate one-move position demonstrates `game_over`. |
| 3.2 · Observation experiment | Four 15-step trajectories compare board-only and board-plus-legal-moves observations. | The formatter produces both views of the same opening position. |
| 3.3–3.5 · Simulation, Python, and skill | Recorded snippets searched with `simulate_move` and committed one move with `play_move`. | A one-ply simulation, a skill lookup, and the sandbox command and live-board refresh. |

The sandbox expiration prompted a circuit breaker in `ChessAgent`. Recoverable move errors go back to the model. After repeated connection failures and a final failed state read, the harness stops before another model call. The committed [learning log](https://github.com/xqzr2000/cmu-11-768-assignment-1/blob/ca87bbef76059287468d82fb37ea01aa9646c31d/docs/Learning-Progress.md) also records the OpenCode experiment and the earlier trajectory-logging gap.

### Read the implementation from this notebook

Choose an entry below and rerun the cell to print the checked-out method with source line numbers. Try `"prompt"` to find where the working memory is inserted or `"chess tools"` to find the rule that permits only one board-changing call per response.

In [ ]:
methods = {
    "prompt": Agent.build_prompt,
    "ReAct": Agent.run,
    "coding tools": CodeAgent.execute_tool_calls,
    "compaction": Agent.compact_context,
    "chess tools": ChessAgent.execute_tool_calls,
    "sandbox refresh": ChessAgent._run_python,
}

# Change this value to read another method.
method = methods["prompt"]
lines, first_line = inspect.getsourcelines(method)
print(f"{method.__module__}.{method.__qualname__}\n")
print("".join(f"{number:>4}  {line}" for number, line in enumerate(lines, first_line)))